In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Install required packages
!pip install pypdf PyMuPDF langchain-community rank_bm25 faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [12]:
!pip install langchain-huggingface

In [5]:
cd /content/drive/MyDrive/rag

/content/drive/MyDrive/rag


In [8]:
import os
import sys

# Original path append replaced for Colab compatibility

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

### Read Document

In [9]:
# Download required data files
import os
os.makedirs('data', exist_ok=True)

# Download the PDF document used in this notebook
!wget -O data/attention-is-all-you-need.pdf https://papers.neurips.cc/paper/7181-attention-is-all-you-need.pdf

--2026-03-03 10:08:25--  https://papers.neurips.cc/paper/7181-attention-is-all-you-need.pdf
Resolving papers.neurips.cc (papers.neurips.cc)... 198.202.70.94
Connecting to papers.neurips.cc (papers.neurips.cc)|198.202.70.94|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf [following]
--2026-03-03 10:08:26--  https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf
Resolving proceedings.neurips.cc (proceedings.neurips.cc)... 198.202.70.94
Connecting to proceedings.neurips.cc (proceedings.neurips.cc)|198.202.70.94|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 569417 (556K) [application/pdf]
Saving to: ‘data/attention-is-all-you-need.pdf’

data/attention-is-a 100%[===================>] 556.07K   734KB/s    in 0.8s    

2026-03-03 10:08:28 (734 KB/s) - ‘data/attention-is-all-you-need.pdf’ 

In [10]:
path = "data/attention-is-all-you-need.pdf"

# Splitting Strategies

In [19]:
from typing import List
import re
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    PythonCodeTextSplitter,
    TokenTextSplitter
)
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

class TextSplitLevels:
    @staticmethod
    def character_split(text: str, chunk_size: int = 100) -> List[str]:
        """Naive character splitting"""
        return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

    @staticmethod
    def recursive_split(text: str, chunk_size: int = 1000, chunk_overlap: int = 200) -> List[str]:
        """Recursive character splitting"""
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", " ", ""]
        )
        return splitter.split_text(text)

    @staticmethod
    def document_split(text: str, file_type: str = "markdown") -> List[str]:
        """Document-specific splitting"""
        if file_type.lower() == "markdown":
            headers = [
                ("#", "Header 1"),
                ("##", "Header 2"),
                ("###", "Header 3"),
                ("####", "Header 4")
            ]
            splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers)
            return splitter.split_text(text)
        elif file_type.lower() == "python":
            splitter = PythonCodeTextSplitter(chunk_size=1000, chunk_overlap=200)
            return splitter.split_text(text)
        else:
            # Default to recursive for unknown types
            return TextSplitLevels.recursive_split(text) # Corrected this line to use recursive_split

    @staticmethod
    def semantic_split(text: str, chunk_size: int = 1000, threshold: float = 0.7) -> List[str]:
        """Semantic chunking using embeddings"""
        sentences = sent_tokenize(text)
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

        # Get embeddings for sentences
        sentence_embeddings = embeddings.embed_documents(sentences)

        chunks = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            # Compare similarity between last sentence in chunk and new sentence
            similarity = np.dot(
                sentence_embeddings[-1],
                sentence_embeddings[i]
            ) / (np.linalg.norm(sentence_embeddings[-1]) * np.linalg.norm(sentence_embeddings[i]))

            if similarity > threshold and len(" ".join(current_chunk + [sentences[i]])) < chunk_size:
                current_chunk.append(sentences[i])
            else:
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentences[i]]

        if current_chunk:
            chunks.append(" ".join(current_chunk))

        return chunks

    @staticmethod
    def agentic_split(text: str, max_chunk_size: int = 1000) -> List[str]:
        """Agentic splitting (rule-based + heuristics)"""
        # First split into paragraphs
        paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

        chunks = []
        for para in paragraphs:
            if len(para) <= max_chunk_size:
                chunks.append(para)
            else:
                # Split long paragraphs by sentences, then group
                sentences = sent_tokenize(para)
                current_chunk = []

                for sent in sentences:
                    if len(" ".join(current_chunk + [sent])) <= max_chunk_size:
                        current_chunk.append(sent)
                    else:
                        if current_chunk:
                            chunks.append(" ".join(current_chunk))
                        current_chunk = [sent]

                if current_chunk:
                    chunks.append(" ".join(current_chunk))

        return chunks

def split_text_all(text: str, **kwargs) -> dict:
    """
    Single function containing all of text splitting.

    Args:
        text: Input text to split
        **kwargs: Parameters for each splitting method

    Returns:
        Dictionary with all splitting results
    """
    return {
        "character": TextSplitLevels.character_split(text, kwargs.get('chunk_size', 100)),
        "recursive": TextSplitLevels.recursive_split(
            text,
            kwargs.get('chunk_size', 1000),
            kwargs.get('chunk_overlap', 200)
        ),
        "document": TextSplitLevels.document_split(
            text,
            kwargs.get('file_type', 'markdown')
        ),
        "semantic": TextSplitLevels.semantic_split(
            text,
            kwargs.get('chunk_size', 1000),
            kwargs.get('threshold', 0.7)
        ),
        "_agentic": TextSplitLevels.agentic_split(
            text,
            kwargs.get('max_chunk_size_l5', 1000)
        )
    }

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [18]:
# Example usage
if __name__ == "__main__":
    sample_text = """
    # Header 1
    This is a sample document with multiple sections.

    ## Header 2
    Here's some detailed content that spans multiple sentences.
    It demonstrates how different splitting strategies work.

    def hello_world():
        print("Hello, World!")
        return True

    ### Header 3
    Final section with code and regular text mixed together.
    """

    results = split_text_all(sample_text)

    for level, chunks in results.items():
        print(f"\n{level.upper()}:")
        for i, chunk in enumerate(chunks[:2]):  # Show first 2 chunks
            print(f"  {i+1}: {chunk[:100]}...")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



CHARACTER:
  1: 
    # Header 1
    This is a sample document with multiple sections.
    
    ## Header 2
    Here'...
  2: s some detailed content that spans multiple sentences. 
    It demonstrates how different splitting ...

RECURSIVE:
  1: # Header 1
    This is a sample document with multiple sections.
    
    ## Header 2
    Here's som...

DOCUMENT:


TypeError: 'Document' object is not subscriptable

### Encode document

In [13]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

def encode_pdf(path, chunk_size=1000, chunk_overlap=200):
    """
    Encodes a PDF into a FAISS vector store using open-source Hugging Face embeddings.

    Args:
        path: Path to the PDF file.
        chunk_size: Size of each text chunk.
        chunk_overlap: Overlap between chunks.

    Returns:
        FAISS vector store with encoded content.
    """
    # Load PDF
    loader = PyPDFLoader(path)
    documents = loader.load()

    # Split into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
    )
    texts = text_splitter.split_documents(documents)
    # cleaned_texts = replace_t_with_space(texts)  # Your custom cleaning

    # Open-source embeddings (no API key needed)
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    # Create vector store
    vectorstore = FAISS.from_documents(texts, embeddings)

    return vectorstore


In [14]:
chunks_vector_store = encode_pdf(path, chunk_size=1000, chunk_overlap=200)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Create retriever

In [ ]:
chunks_query_retriever = chunks_vector_store.as_retriever(search_kwargs={"k": 2})

### Test retriever

In [ ]:
test_query = "What is transformer model?"
context = retrieve_context_per_question(test_query, chunks_query_retriever)
show_context(context)

Context 1:
Chapter 2: Causes of Climate Change 
Greenhouse Gases 
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous 
oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is essential 
for life on Earth, as it keeps the planet warm enough to support life. However, human 
activities have intensified this natural process, leading to a warmer climate. 
Fossil Fuels 
Burning fossil fuels for energy releases large amounts of CO2. This includes coal, oil, and 
natural gas used for electricity, heating, and transportation. The industrial revolution marked 
the beginning of a significant increase in fossil fuel consumption, which continues to rise 
today. 
Coal


Context 2:
Most of these climate changes are attributed to very small variations in Earth's orbit that 
change the amount of solar energy our planet receives. During the Holocene epoch,

/content/RAG_TECHNIQUES/helper_functions.py:143: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = chunks_query_retriever.get_relevant_documents(question)


### Evaluate results

In [ ]:
#Note - this currently works with OPENAI only
evaluate_rag(chunks_query_retriever)

{'questions': ['1. **Multiple Choice: Causes of Climate Change**',
  '   - What is the primary cause of the current climate change trend?',
  '     A) Solar radiation variations',
  '     B) Natural cycles of the Earth',
  '     C) Human activities, such as burning fossil fuels',
  '     D) Volcanic eruptions',
  '',
  '2. **True or False: Climate Change Impacts**',
  '   - True or False: Climate change only affects the temperature of the planet, not weather patterns, sea levels, or ecosystems.',
  '',
  '3. **Short Answer: Mitigation Strategies**',
  '   - Describe two effective strategies that could be implemented to mitigate the effects of climate change.',
  '',
  '4. **Matching: Climate Change Terminology**',
  '   - Match the following terms with their correct definitions:',
  '     A) Greenhouse Gases',
  '     B) Carbon Footprint',
  '     C) Renewable Energy',
  '     D) Adaptation',
  '     - Definitions:',
  '       1. The total amount of greenhouse gases produced to directl

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--simple-rag)